# Ejercicio 4

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

from lista4.data import load_csv
from lista4.models import fit_logistic
from lista4.metrics import riesgo_logistico, errores_clasificacion, clasificar

In [2]:
df = load_csv("clientes.csv")
df.head()

,antiguedad_meses,uso_30d_previos,facturas_vencidas_180d,tickets_90d,pago_automatico,ajuste_comercial_30d,abandona_30d
0,66,37.927153,0,1,0,23.039078,0
1,12,46.656223,0,1,1,19.158849,0
2,49,22.990662,0,3,1,27.712278,0
3,49,24.546478,1,0,0,96.997980,1
4,30,64.413222,0,2,1,0.000000,0


## (a) Variables disponibles en t0

In [3]:
# antiguedad_meses: disponible en t0
# uso_30d_previos: usa datos de t0-30 a t0 -> disponible en t0
# facturas_vencidas_180d: usa datos de t0-180 a t0 -> disponible en t0
# tickets_90d: usa datos de t0-90 a t0 -> disponible en t0
# pago_automatico: disponible en t0
# ajuste_comercial_30d: usa datos de t0 a t0+30 -> AUN NO disponible en t0 (es informacion futura)

variables_disponibles = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico",
]
variables_disponibles

['antiguedad_meses',
 'uso_30d_previos',
 'facturas_vencidas_180d',
 'tickets_90d',
 'pago_automatico']

## (b) Modelos M1 (todas las variables) y M2 (solo variables_disponibles)

In [4]:
todas_las_variables = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico",
    "ajuste_comercial_30d",
]

y = df["abandona_30d"].values
X_M1 = df[todas_las_variables].values
X_M2 = df[variables_disponibles].values

M1 = fit_logistic(X_M1, y)
M2 = fit_logistic(X_M2, y)

p_M1 = M1.predict_proba(X_M1)[:, 1]
p_M2 = M2.predict_proba(X_M2)[:, 1]

## (c)-(d) Riesgo empírico, errores de clasificación y tabla

In [5]:
clase_M1 = clasificar(p_M1, umbral=0.5)
clase_M2 = clasificar(p_M2, umbral=0.5)

R_M1 = riesgo_logistico(y, p_M1)
R_M2 = riesgo_logistico(y, p_M2)

n_err_M1, prop_err_M1 = errores_clasificacion(y, clase_M1)
n_err_M2, prop_err_M2 = errores_clasificacion(y, clase_M2)

tabla_4 = pd.DataFrame({
    "modelo": ["M1 (todas las variables)", "M2 (solo variables_disponibles)"],
    "riesgo_empirico_logistico": [R_M1, R_M2],
    "n_errores_clasificacion": [n_err_M1, n_err_M2],
    "proporcion_errores": [prop_err_M1, prop_err_M2],
})
tabla_4

,modelo,riesgo_empirico_logistico,n_errores_clasificacion,proporcion_errores
0,M1 (todas las variables),0.342485,160,0.133333
1,M2 (solo variables_disponibles),0.571716,336,0.280000


## (e) ¿Cuál modelo puede ejecutarse en t0?

In [6]:
print("M1 usa 'ajuste_comercial_30d', que se construye con datos de t0 a t0+30:")
print("  -> esa informacion NO existe todavia en el momento t0, por lo tanto M1")
print("     no se puede ejecutar en t0 (usaria informacion del futuro).")
print()
print("M2 solo usa variables_disponibles (conocidas en t0), por lo tanto")
print("es el modelo que SI puede usarse en produccion en el momento t0.")
print()
print("La variable que produce la diferencia entre M1 y M2 es: 'ajuste_comercial_30d'")

M1 usa 'ajuste_comercial_30d', que se construye con datos de t0 a t0+30:
  -> esa informacion NO existe todavia en el momento t0, por lo tanto M1
     no se puede ejecutar en t0 (usaria informacion del futuro).

M2 solo usa variables_disponibles (conocidas en t0), por lo tanto
es el modelo que SI puede usarse en produccion en el momento t0.

La variable que produce la diferencia entre M1 y M2 es: 'ajuste_comercial_30d'
